# 25 · Multiscale Boundary Failure Persistence

This notebook tests whether the boundary-failure story from Notebook 24 is **structural** rather than accidental.

## Core question

Does zeta-vs-GUE boundary failure persist across:
- window size
- feature family
- height block
- sample size
- model class

while Poisson comparison tasks remain separable?

## Tasks

We compare:
1. **zeta vs GUE**
2. **zeta vs Poisson**
3. **GUE vs Poisson**

## Metrics

For each experiment, we track:
- test accuracy
- decision-score overlap
- distance-to-boundary overlap

## Main purpose

If zeta-vs-GUE remains hard while Poisson tasks remain easy across these sweeps, that supports the claim that the failure is a persistent structural property of the local statistics.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Descriptor builders

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_feature_matrix(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    if feature_family == "minimal":
        X = np.array([[window_entropy(w), unevenness(w), ratio_mean(w)] for w in windows_n], dtype=float)
    elif feature_family == "summary_only":
        X = np.array([[window_entropy(w), unevenness(w), reversal_symmetry_score(w)] for w in windows_n], dtype=float)
    elif feature_family == "ratio_only":
        X = np.array([[ratio_mean(w), ratio_std(w)] for w in windows_n], dtype=float)
    elif feature_family == "full":
        X = np.array([[window_entropy(w), unevenness(w), reversal_symmetry_score(w), ratio_mean(w), ratio_std(w)] for w in windows_n], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, X

## Model helpers

In [ ]:
def split_train_test(X, frac=0.6):
    n = len(X)
    cut = int(frac * n)
    return X[:cut], X[cut:]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def predict_logistic(X, w):
    return (predict_proba_logistic(X, w) >= 0.5).astype(int)

def quadratic_features(X):
    cols = [X[:, i] for i in range(X.shape[1])]
    out = cols[:]
    for i in range(X.shape[1]):
        out.append(cols[i] * cols[i])
    for i in range(X.shape[1]):
        for j in range(i + 1, X.shape[1]):
            out.append(cols[i] * cols[j])
    return np.column_stack(out)

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

## Unified evaluator

In [ ]:
def evaluate_pair(X0, X1, model_class="rbf"):
    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    X0_train, X0_test = split_train_test(X0, frac=0.6)
    X1_train, X1_test = split_train_test(X1, frac=0.6)

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))

    if model_class == "linear":
        Xtr, Xte = standardize_train_test(X_train, X_test)
        w = fit_logistic_regression(Xtr, y_train, lr=0.12, n_steps=3000, reg=1e-4)
        scores = decision_function_logistic(Xte, w)
        probs = predict_proba_logistic(Xte, w)
    elif model_class == "quadratic":
        Q_train = quadratic_features(X_train)
        Q_test = quadratic_features(X_test)
        Xtr, Xte = standardize_train_test(Q_train, Q_test)
        w = fit_logistic_regression(Xtr, y_train, lr=0.08, n_steps=3500, reg=1e-4)
        scores = decision_function_logistic(Xte, w)
        probs = predict_proba_logistic(Xte, w)
    elif model_class == "rbf":
        Xtr_std, Xte_std = standardize_train_test(X_train, X_test)
        prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
        gamma = estimate_rbf_gamma(Xtr_std)
        R_train = rbf_features(Xtr_std, prototypes, gamma)
        R_test = rbf_features(Xte_std, prototypes, gamma)
        w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
        scores = decision_function_logistic(R_test, w)
        probs = predict_proba_logistic(R_test, w)
    else:
        raise ValueError(f"Unknown model class: {model_class}")

    preds = (probs >= 0.5).astype(int)
    acc = accuracy(y_test, preds)

    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]
    dists = np.abs(scores)
    dists0 = dists[y_test == 0]
    dists1 = dists[y_test == 1]

    return {
        "accuracy": acc,
        "score_overlap": overlap_coefficient_from_hist(scores0, scores1, bins=30),
        "distance_overlap": overlap_coefficient_from_hist(dists0, dists1, bins=30),
    }

## Sweep definitions

In [ ]:
window_sizes = [3, 4, 5, 6, 7]
feature_families = ["minimal", "summary_only", "ratio_only", "full", "raw_window"]
height_blocks = [(0, 400), (400, 800), (800, 1200), (1200, 1600)]
sample_sizes = [200, 400, 600]
model_classes = ["linear", "quadratic", "rbf"]

## Main multiscale RBF sweep

In [ ]:
results = []

for k in window_sizes:
    for feature_family in feature_families:
        for (start, stop) in height_blocks:
            zeta_g = zeta_gaps_full[start:stop]
            poisson_g = poisson_gaps_full[start:stop]
            gue_needed = max(stop - start + 300, 900)
            gue_g = gue_gaps_full[:gue_needed]

            _, zeta_X_all = build_feature_matrix(zeta_g, feature_family=feature_family, k=k)
            _, poisson_X_all = build_feature_matrix(poisson_g, feature_family=feature_family, k=k)
            _, gue_X_all = build_feature_matrix(gue_g, feature_family=feature_family, k=k)

            for n in sample_sizes:
                m = min(len(zeta_X_all), len(poisson_X_all), len(gue_X_all), n)
                if m < 40:
                    continue

                zeta_X = zeta_X_all[:m]
                poisson_X = poisson_X_all[:m]
                gue_X = gue_X_all[:m]

                for task_name, A, B in [
                    ("zeta_vs_GUE", zeta_X, gue_X),
                    ("zeta_vs_Poisson", zeta_X, poisson_X),
                    ("GUE_vs_Poisson", gue_X, poisson_X),
                ]:
                    out = evaluate_pair(A, B, model_class="rbf")
                    results.append({
                        "task": task_name,
                        "k": k,
                        "feature_family": feature_family,
                        "height_block": f"{start+1}-{stop}",
                        "sample_size": m,
                        "model_class": "rbf",
                        **out,
                    })

len(results)

## Representative model-class comparison

In [ ]:
model_compare = []

k0 = 5
feature0 = "minimal"
start0, stop0 = height_blocks[0]

zeta_g = zeta_gaps_full[start0:stop0]
poisson_g = poisson_gaps_full[start0:stop0]
gue_g = gue_gaps_full[:max(stop0 - start0 + 300, 900)]

_, zeta_X_all = build_feature_matrix(zeta_g, feature_family=feature0, k=k0)
_, poisson_X_all = build_feature_matrix(poisson_g, feature_family=feature0, k=k0)
_, gue_X_all = build_feature_matrix(gue_g, feature_family=feature0, k=k0)

m = min(len(zeta_X_all), len(poisson_X_all), len(gue_X_all), 400)
zeta_X = zeta_X_all[:m]
poisson_X = poisson_X_all[:m]
gue_X = gue_X_all[:m]

for model_class in model_classes:
    for task_name, A, B in [
        ("zeta_vs_GUE", zeta_X, gue_X),
        ("zeta_vs_Poisson", zeta_X, poisson_X),
        ("GUE_vs_Poisson", gue_X, poisson_X),
    ]:
        out = evaluate_pair(A, B, model_class=model_class)
        model_compare.append({
            "task": task_name,
            "model_class": model_class,
            **out,
        })

model_compare

## Accuracy vs window size

In [ ]:
def aggregate_metric(records, task, metric, fixed_feature="minimal", fixed_height="1-400", fixed_n=400):
    xs, ys = [], []
    for k in window_sizes:
        vals = [
            r[metric] for r in records
            if r["task"] == task and r["k"] == k and r["feature_family"] == fixed_feature
            and r["height_block"] == fixed_height and r["sample_size"] == fixed_n
        ]
        if vals:
            xs.append(k)
            ys.append(float(np.mean(vals)))
    return xs, ys

plt.figure(figsize=(9.8, 4.8))
for task in ["zeta_vs_GUE", "zeta_vs_Poisson", "GUE_vs_Poisson"]:
    xs, ys = aggregate_metric(results, task, "accuracy", fixed_feature="minimal", fixed_height="1-400", fixed_n=400)
    plt.plot(xs, ys, marker="o", label=task)
plt.ylim(0.45, 1.0)
plt.xlabel("window size k")
plt.ylabel("RBF accuracy")
plt.title("Accuracy vs window size")
plt.legend()
plt.tight_layout()
plt.show()

## Score overlap vs window size

In [ ]:
plt.figure(figsize=(9.8, 4.8))
for task in ["zeta_vs_GUE", "zeta_vs_Poisson", "GUE_vs_Poisson"]:
    xs, ys = aggregate_metric(results, task, "score_overlap", fixed_feature="minimal", fixed_height="1-400", fixed_n=400)
    plt.plot(xs, ys, marker="o", label=task)
plt.xlabel("window size k")
plt.ylabel("score overlap")
plt.title("Decision-score overlap vs window size")
plt.legend()
plt.tight_layout()
plt.show()

## Accuracy by feature family

In [ ]:
def aggregate_by_feature(records, task, metric, fixed_k=5, fixed_height="1-400", fixed_n=400):
    vals = []
    for ff in feature_families:
        subset = [
            r[metric] for r in records
            if r["task"] == task and r["k"] == fixed_k and r["feature_family"] == ff
            and r["height_block"] == fixed_height and r["sample_size"] == fixed_n
        ]
        vals.append(float(np.mean(subset)) if subset else np.nan)
    return vals

x = np.arange(len(feature_families))
width = 0.25

vals1 = aggregate_by_feature(results, "zeta_vs_GUE", "accuracy")
vals2 = aggregate_by_feature(results, "zeta_vs_Poisson", "accuracy")
vals3 = aggregate_by_feature(results, "GUE_vs_Poisson", "accuracy")

plt.figure(figsize=(11.5, 4.8))
plt.bar(x - width, vals1, width, label="zeta vs GUE")
plt.bar(x, vals2, width, label="zeta vs Poisson")
plt.bar(x + width, vals3, width, label="GUE vs Poisson")
plt.xticks(x, feature_families, rotation=20)
plt.ylim(0.45, 1.0)
plt.ylabel("RBF accuracy")
plt.title("Accuracy by feature family")
plt.legend()
plt.tight_layout()
plt.show()

## Score overlap by feature family

In [ ]:
vals1 = aggregate_by_feature(results, "zeta_vs_GUE", "score_overlap")
vals2 = aggregate_by_feature(results, "zeta_vs_Poisson", "score_overlap")
vals3 = aggregate_by_feature(results, "GUE_vs_Poisson", "score_overlap")

plt.figure(figsize=(11.5, 4.8))
plt.bar(x - width, vals1, width, label="zeta vs GUE")
plt.bar(x, vals2, width, label="zeta vs Poisson")
plt.bar(x + width, vals3, width, label="GUE vs Poisson")
plt.xticks(x, feature_families, rotation=20)
plt.ylabel("score overlap")
plt.title("Decision-score overlap by feature family")
plt.legend()
plt.tight_layout()
plt.show()

## Accuracy vs height block

In [ ]:
def aggregate_by_height(records, task, metric, fixed_k=5, fixed_feature="minimal", fixed_n=400):
    xs, ys = [], []
    for (start, stop) in height_blocks:
        label = f"{start+1}-{stop}"
        subset = [
            r[metric] for r in records
            if r["task"] == task and r["k"] == fixed_k and r["feature_family"] == fixed_feature
            and r["height_block"] == label and r["sample_size"] == fixed_n
        ]
        if subset:
            xs.append(label)
            ys.append(float(np.mean(subset)))
    return xs, ys

plt.figure(figsize=(10.2, 4.8))
for task in ["zeta_vs_GUE", "zeta_vs_Poisson", "GUE_vs_Poisson"]:
    xs, ys = aggregate_by_height(results, task, "accuracy", fixed_k=5, fixed_feature="minimal", fixed_n=400)
    plt.plot(xs, ys, marker="o", label=task)
plt.ylim(0.45, 1.0)
plt.ylabel("RBF accuracy")
plt.title("Accuracy vs height block")
plt.legend()
plt.tight_layout()
plt.show()

## Accuracy vs sample size

In [ ]:
def aggregate_by_sample(records, task, metric, fixed_k=5, fixed_feature="minimal", fixed_height="1-400"):
    xs, ys = [], []
    for n in sample_sizes:
        subset = [
            r[metric] for r in records
            if r["task"] == task and r["k"] == fixed_k and r["feature_family"] == fixed_feature
            and r["height_block"] == fixed_height and r["sample_size"] == n
        ]
        if subset:
            xs.append(n)
            ys.append(float(np.mean(subset)))
    return xs, ys

plt.figure(figsize=(9.8, 4.8))
for task in ["zeta_vs_GUE", "zeta_vs_Poisson", "GUE_vs_Poisson"]:
    xs, ys = aggregate_by_sample(results, task, "accuracy", fixed_k=5, fixed_feature="minimal", fixed_height="1-400")
    plt.plot(xs, ys, marker="o", label=task)
plt.ylim(0.45, 1.0)
plt.xlabel("sample size")
plt.ylabel("RBF accuracy")
plt.title("Accuracy vs sample size")
plt.legend()
plt.tight_layout()
plt.show()

## Representative model-class comparison

In [ ]:
tasks = ["zeta_vs_GUE", "zeta_vs_Poisson", "GUE_vs_Poisson"]
x = np.arange(len(tasks))
width = 0.24

linear_vals = [next(r["accuracy"] for r in model_compare if r["task"] == t and r["model_class"] == "linear") for t in tasks]
quad_vals = [next(r["accuracy"] for r in model_compare if r["task"] == t and r["model_class"] == "quadratic") for t in tasks]
rbf_vals = [next(r["accuracy"] for r in model_compare if r["task"] == t and r["model_class"] == "rbf") for t in tasks]

plt.figure(figsize=(10.5, 4.8))
plt.bar(x - width, linear_vals, width, label="linear")
plt.bar(x, quad_vals, width, label="quadratic")
plt.bar(x + width, rbf_vals, width, label="rbf")
plt.xticks(x, tasks, rotation=20)
plt.ylim(0.45, 1.0)
plt.ylabel("accuracy")
plt.title("Representative model-class comparison")
plt.legend()
plt.tight_layout()
plt.show()

## Heatmap-like matrices of mean score overlap

In [ ]:
def build_overlap_matrix(records, task, fixed_feature="minimal", fixed_n=400):
    M = np.full((len(window_sizes), len(height_blocks)), np.nan)
    for i, k in enumerate(window_sizes):
        for j, (start, stop) in enumerate(height_blocks):
            label = f"{start+1}-{stop}"
            vals = [
                r["score_overlap"] for r in records
                if r["task"] == task and r["k"] == k and r["feature_family"] == fixed_feature
                and r["height_block"] == label and r["sample_size"] == fixed_n
            ]
            if vals:
                M[i, j] = float(np.mean(vals))
    return M

M_zg = build_overlap_matrix(results, "zeta_vs_GUE")
M_zp = build_overlap_matrix(results, "zeta_vs_Poisson")
M_gp = build_overlap_matrix(results, "GUE_vs_Poisson")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)

for ax, M, title in zip(axes, [M_zg, M_zp, M_gp], ["zeta vs GUE", "zeta vs Poisson", "GUE vs Poisson"]):
    im = ax.imshow(M, aspect="auto", origin="lower", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks(np.arange(len(height_blocks)), [f"{a+1}-{b}" for a, b in height_blocks], rotation=20)
    ax.set_yticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_xlabel("height block")
    ax.set_ylabel("window size k")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="mean score overlap")
plt.tight_layout()
plt.show()

## Compact summary

In [ ]:
summary = {
    "n_records": len(results),
    "window_sizes": window_sizes,
    "feature_families": feature_families,
    "height_blocks": [f"{a+1}-{b}" for a, b in height_blocks],
    "sample_sizes": sample_sizes,
    "model_compare": model_compare,
    "mean_overlap_default": {
        "zeta_vs_GUE": float(np.nanmean(M_zg)),
        "zeta_vs_Poisson": float(np.nanmean(M_zp)),
        "GUE_vs_Poisson": float(np.nanmean(M_gp)),
    },
}
summary

## Notes

- If zeta-vs-GUE stays near low accuracy and high overlap across these sweeps, while the Poisson tasks stay at high accuracy and low overlap, then the failure is multiscale and representation-robust.
- The main multiscale sweep uses the strongest nonlinear family (RBF) to keep runtime reasonable; the representative model comparison checks that the conclusion is not specific to one boundary class.
- This notebook emphasizes persistence, not optimization. A future notebook could do hyperparameter sweeps if needed.

## Next directions

A natural next notebook could do one of these:

1. bootstrap the multiscale overlap matrices  
2. extend the same persistence test to conditional windows  
3. move to longer-range or multi-window features  
4. compare calibrated confidence rather than hard accuracy